# Introducing Cats

Common abstractions for functional programming in **Scala**

## What is Cats?

![cats](https://typelevel.org/cats/img/cats2.png)

 - Library providing *abstractions* for functional programming in Scala
 - Foundation for an ecosystem of *pure*, *typeful* libraries
 - Why **cats**?
     - Concepts coming from *“category theory”*

## What Cats offers

 - range of type classes
 - set of extension methods 
 - general FP primitives
 - (among many other things)

**Purpose:** allow to write general and *extensible* FP code

## How to include Cats in a project?

Including in an `build.sbt` project:

```
ThisBuild / version := "1.1"
ThisBuild / scalaVersion := "3.3.7"
ThisBuild / organization := "ch.hevs"
  libraryDependencies ++= Seq(
    "org.typelevel" %% "cats-core" % "2.13.0" ,
    "org.scalatest" %% "scalatest" % "3.2.14" % Test
)
```

## Importing Cats in this notebook

In [ ]:
import $ivy.`org.typelevel::cats-core:2.13.0`

## Using Type Classes

Cats relies heavily on Type Classes (remember?)

 - enable ad-hoc polymorphism, i.e., overloading
 - alternative to subtyping
 - parametric polymorphism + ad-hoc polymorphism.



Example: remeber `Taxable[A]`:

In [ ]:
trait Taxable[A]:
  extension(a:A)
    def computeTax:Double

In [ ]:
case class Architect(name: String, income:Double)

In [ ]:
given Taxable[Architect] with
  extension(a:Architect) def computeTax = a.income * 0.3

## Show

The `toString` method has several problems, e.g. can be called on anything, and do unexpected things:

In [ ]:
34.5.toString

In [ ]:
(new Architect("jim",34)).toString

In [ ]:
(new {val a=34}).toString

**Show** provides an alternative to `toString`, only to apply to desired classes.

The signature of the `show` function is given as:
```
def show[A](f: A => String): Show[A]
```

We can use `Show` applying it to a given type:

In [ ]:
import cats.Show

In [ ]:
val showInt = Show.apply[Int]
showInt.show(4)

In [ ]:
Show[Int].show(4)

We can use it for different types, which are of course checked:

In [ ]:
Show[Boolean].show(4)

In [ ]:
Show[Boolean].show(true)

Cats also has lots of syntax sugar helpers that make things even simpler:

In [ ]:
import cats.syntax.show.toShow

And now we can use `show` as an extension method:

In [ ]:
4.show

In [ ]:
false.show

In [ ]:
List(4,8,7).show

We can also define a custom behavior for show, for example for the `Boolean` type:

In [ ]:
given Show[Boolean] with
  def show(b: Boolean): String = b match
    case true => "vrai"
    case false => "faux"

In [ ]:
false.show

A simpler syntax is also possible:

In [ ]:
given Show[Double] = Show.show(d => s"$d*10^0")

In [ ]:
39.45.show

In [ ]:
39.4f.show

Surely we can do it for other types as well:

In [ ]:
case class Student(name:String, age:Int)

In [ ]:
given Show[Student] = 
  Show.show(s => s"${s.name}, ${s.age} years old")

In [ ]:
val will = Student("will",21)
will.show

## Equal

Equality with a bit of type checking

Consider we want to test this equality:

In [ ]:
4 == true

We cannot even compile it as the types do not match. But if a supertype is used then types can often be blurred, and we compare even when it makes no sense:

In [ ]:
(4:Any) == true

For other types this can lead to useless comparisons. 

Consider these `Doctor` and `Nurse` classes:

In [ ]:
case class Doctor(name:String)
case class Nurse(name:String)

In [ ]:
val doris = Doctor("doris")
val doris2 = Doctor("Doris")
val nick = Nurse("nick")

Comparison between a doctor and nurse will always be false as the types are not even the same:

In [ ]:
doris == nick

This can be even more problematic in lists. Consider a list of `Option`

In [ ]:
val list:List[Any] = List(8,21,6).map(Option(_))

Then when we try to do a filter, we might not get what we expect:

In [ ]:
list filter (i => i == 6)

Same on a list of `Doctor` or `Nurse`:

In [ ]:
val list2 = List(doris,nick).map(Option(_))

In [ ]:
list2.filter(p => p == nick)

### Introducing Eq

The Cats `Eq` provides some stronger type equality:

In [ ]:
import cats.Eq
Eq[Int].eqv(123,123)

Now we can have compile-time errors:

In [ ]:
Eq[Int].eqv(123,"123")

Syntax is not the simplest, but cats as always brings some syntax coating to make things easier:

In [ ]:
import cats.syntax.eq._

123 === 123

In [ ]:
123 =!= 234

### Custom equality

Now we can use `Eq` to define equality in different ways.

For example we can define it on lower-case name comparison:

In [ ]:
given Eq[Doctor] = 
  Eq.instance[Doctor]{(a,b)=>a.name.toLowerCase === b.name.toLowerCase}

Now we can compare two Doctors:

In [ ]:
doris === doris2

Type is checked anyway if we try to compare with a `Nurse`:

In [ ]:
doris === nick

### Scala strict equality

Without Cats there are some type safety equality utils in the Scala lib:

In [ ]:
import scala.language.strictEquality

doris == nick

We can allow comparison between two types, but it has to be made explicit:

In [ ]:
given CanEqual[Doctor,Nurse] = CanEqual.derived

In [ ]:
doris == nick

This works on one way only:

In [ ]:
nick == doris

In [ ]:
given CanEqual[Nurse,Doctor] = CanEqual.derived

In [ ]:
nick == doris